# STEP 0 - Validating the sector definition: drawing the sample

This notebook draws the hand-labelling sample described in `docs/validation.md`.
It does three things and nothing else:

1. patents of invention, earliest priority year 2000-2022 were considered;
2. splits it into the three strata of the protocol and draws 20 / 20 / 60 families with a fixed seed;
3. attaches a title and abstract to each family and writes the files the team labels.
4. Hand edit labels by two raters
5. Compare the results of two raters
6. Record the answers

Precision is computed later, once the labels are in. Nothing here changes the definition.

## 1. Setup

In [ ]:
import importlib.util
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from sqlalchemy import func

REPO = Path.cwd() if (Path.cwd() / "config").is_dir() else Path.cwd().parent
sys.path.insert(0, str(REPO / "src"))

from config import load_config

from epo.tipdata.patstat import PatstatClient
from epo.tipdata.patstat.database.models import (
    TLS201_APPLN,
    TLS202_APPLN_TITLE,
    TLS203_APPLN_ABSTR,
)

# src/01_extract.py holds the sector query. Its name starts with a digit, so it
# cannot be loaded with a plain import statement.
_spec = importlib.util.spec_from_file_location("extract", REPO / "src" / "01_extract.py")
extract = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(extract)

cfg = load_config()

## Parameters
ENV = cfg.env
SEED = 30 
SAMPLE_SIZES = {"A": 20, "B": 20, "C": 60}
OUT_DIR = REPO / "outputs" / "tables"
OUT_DIR.mkdir(parents=True, exist_ok=True)

patstat = PatstatClient(env=ENV)
db = patstat.orm()
print(f"Sector: {cfg.sector_name} | env: {ENV} | years {cfg.year_min}-{cfg.year_max} "
      f"| ipr_type {cfg.ipr_type} | seed {SEED}")

## 2. The strata

| Stratum | Families | What it tests |
|---|---|---|
| A | carry a core `G06N10` code | sanity check; expected near-perfect |
| B | carry `H10N60/12` (Josephson devices) but no core code | precision of the enabling-hardware class |
| C | match a keyword but no CPC code at all | precision of the free-text branch |

In [ ]:
def family_ids(query):
    """Execute a family-id query and return its distinct ids as a Python set."""
    return set(patstat.df(query.distinct())["docdb_family_id"].astype(int))


ids_core = family_ids(extract.families_by_cpc(db, cfg.cpc_core))
ids_hardware = family_ids(extract.families_by_cpc(db, cfg.cpc_hardware))
ids_cpc = family_ids(extract.families_by_cpc(db, cfg.cpc_prefixes))
ids_keywords = family_ids(extract.families_by_keywords(db, cfg.keywords))

for label, ids in [("core G06N10", ids_core), ("hardware H10N60/12", ids_hardware),
                   ("CPC branch (all codes)", ids_cpc), ("keyword branch", ids_keywords)]:
    print(f"{label:<26} {len(ids):>10,}   (all years, before the window)")

### 2.1 Restricting to the population the analysis counts

The branch counts reported in the README come from queries with no year or `ipr_type` filter.
The Step 1 figures, however, count only patents of invention whose family's earliest priority
year falls in 2000-2022 (`family_scope` in the Step 1 notebook). We sample from that population,
so the precision we measure applies to the families actually behind the figures.

In [ ]:
first_year = func.min(getattr(TLS201_APPLN, cfg.family_time_column))
in_window = patstat.df(
    db.query(TLS201_APPLN.docdb_family_id.label("docdb_family_id"), first_year.label("year"))
    .filter(
        TLS201_APPLN.ipr_type == cfg.ipr_type,
        TLS201_APPLN.docdb_family_id.in_(extract.sector_family_ids(db, cfg).subquery().select()),
    )
    .group_by(TLS201_APPLN.docdb_family_id)
    .having(first_year.between(cfg.year_min, cfg.year_max))
)
window = set(in_window["docdb_family_id"].astype(int))

strata = {
    "A": ids_core & window,
    "B": (ids_hardware - ids_core) & window,
    "C": (ids_keywords - ids_cpc) & window,
}
sector = (ids_cpc | ids_keywords) & window
not_sampled = sector - strata["A"] - strata["B"] - strata["C"]   # communication codes only

population = pd.DataFrame(
    {
        "description": ["core G06N10", "hardware only", "keyword only",
                        "communication only (scope, not sampled)", "sector, 2000-2022"],
        "families": [len(strata[s]) for s in SAMPLE_SIZES] + [len(not_sampled), len(sector)],
        "sample": [min(n, len(strata[s])) for s, n in SAMPLE_SIZES.items()] + [0, sum(SAMPLE_SIZES.values())],
    },
    index=pd.Index(["A", "B", "C", "comms_only", "sector"], name="stratum"),
)
population["share_of_sector"] = (population["families"] / len(sector)).round(4)
population

## 3. Drawing the sample

Each stratum is sorted before sampling, because the iteration order of a Python set is not
reproducible across sessions. With the ids sorted and the seed fixed, re-running this cell on the
same PATSTAT edition draws the same 100 families.

In [ ]:
rng = np.random.default_rng(SEED)
rows = []
for stratum, n in SAMPLE_SIZES.items():
    pool = np.array(sorted(strata[stratum]))
    picked = rng.choice(pool, size=min(n, len(pool)), replace=False)
    rows += [{"stratum": stratum, "docdb_family_id": int(f)} for f in picked]

sample = pd.DataFrame(rows)
sample["stratum"].value_counts().sort_index()

## 4. Title and abstract for each family

A family has several applications, often in several languages. For each sampled family we take the
earliest-filed member that has an English abstract; failing that, one with an English title; failing
that, the earliest member. Families left without English text are flagged, and are the likeliest
`?` labels.

In [ ]:
members = patstat.df(
    db.query(
        TLS201_APPLN.docdb_family_id,
        TLS201_APPLN.appln_id,
        TLS201_APPLN.appln_auth,
        TLS201_APPLN.appln_nr,
        TLS201_APPLN.appln_filing_date,
        TLS202_APPLN_TITLE.appln_title_lg,
        TLS202_APPLN_TITLE.appln_title,
        TLS203_APPLN_ABSTR.appln_abstract_lg,
        TLS203_APPLN_ABSTR.appln_abstract,
    )
    .outerjoin(TLS202_APPLN_TITLE, TLS201_APPLN.appln_id == TLS202_APPLN_TITLE.appln_id)
    .outerjoin(TLS203_APPLN_ABSTR, TLS201_APPLN.appln_id == TLS203_APPLN_ABSTR.appln_id)
    .filter(TLS201_APPLN.docdb_family_id.in_(sample["docdb_family_id"].tolist()))
)

members["en_abstract"] = members["appln_abstract_lg"].str.strip().eq("en") & members["appln_abstract"].notna()
members["en_title"] = members["appln_title_lg"].str.strip().eq("en") & members["appln_title"].notna()
best = (
    members.sort_values(
        ["docdb_family_id", "en_abstract", "en_title", "appln_filing_date", "appln_id"],
        ascending=[True, False, False, True, True],
    )
    .drop_duplicates("docdb_family_id")
    .rename(columns={"appln_title": "title", "appln_abstract": "abstract"})
)

sample = sample.merge(
    best[["docdb_family_id", "appln_auth", "appln_nr", "appln_filing_date",
          "title", "abstract", "en_title", "en_abstract"]],
    on="docdb_family_id", how="left",
)
print(f"{len(sample)} families | without an English abstract: {(~sample['en_abstract']).sum()}")
sample.head()

## 5. Writing the files

Three files go to `outputs/tables/`:

| File | Content | Who uses it |
|---|---|---|
| `validation_population.csv` | stratum sizes | the precision step, as weights |
| `validation_sample.csv` | the 100 families **with** their stratum | the precision step |
| `validation_to_label.csv` | the same families, shuffled, **without** the stratum, with empty label columns | the two raters |

The raters' sheet hides the stratum on purpose: knowing that a family came in through the keyword
branch invites a harsher reading of it. Labels are joined back to the strata on `docdb_family_id`.

The files are written once. If `validation_sample.csv` already exists the cell stops, so that a
re-run cannot replace a sample the team may already be labelling. Delete the file to draw again.

In [ ]:
SAMPLE_FILE = OUT_DIR / "validation_sample.csv"
LABEL_FILE = OUT_DIR / "validation_to_label.csv"
POPULATION_FILE = OUT_DIR / "validation_population.csv"

if ENV != "PROD":
    print("TEST environment: nothing written. Switch ENV to PROD for the real sample.")
elif SAMPLE_FILE.exists():
    print(f"{SAMPLE_FILE.name} already exists and was not overwritten. Delete it to draw a new sample.")
else:
    population.to_csv(POPULATION_FILE)
    sample.to_csv(SAMPLE_FILE, index=False)

    sheet = (
        sample.sample(frac=1, random_state=SEED)[["docdb_family_id", "title", "abstract"]]
        .assign(rater_1="", rater_2="", final="", note="")
    )
    sheet.insert(0, "row", range(1, len(sheet) + 1))
    sheet.to_csv(LABEL_FILE, index=False, encoding="utf-8-sig")   # utf-8-sig: opens cleanly in Excel

    for path in (POPULATION_FILE, SAMPLE_FILE, LABEL_FILE):
        print(f"written: {path.relative_to(REPO)}")

## 6. Next: labelling

Open `validation_to_label.csv` in a shared sheet. Two team members fill `rater_1` and `rater_2`
independently, without seeing each other's column, using the rule in `docs/validation.md`:

- `1` - quantum computing: qubits, quantum processors, quantum algorithms, error correction, control
  electronics or software whose purpose is quantum computation;
- `0` - not quantum computing;
- `?` - cannot tell from the title and abstract.

Disagreements are then settled together into `final`; where both raters agree, `final` may be
left blank and is filled in automatically. Save the finished sheet as CSV under
`outputs/tables/validation_labels.csv` and run section 7.

## 7. Results: precision of the definition

Run this after labelling. It reads the three CSV files written in
section 5 plus the labelled sheet, so it can be run on TIP or on any laptop, without running
section 1 first. The arithmetic lives in `src/validation.py` and is unit-tested in
`tests/test_validation.py`.

What it reports, per `docs/validation.md`:

| Output | Meaning |
|---|---|
| precision per stratum | share of judged families (`1` or `0`) that really are quantum computing, with a 95% Wilson interval |
| weighted precision | the strata recombined by their size in the sector, $p = \sum N_s p_s / \sum N_s$ |
| `?` count | families nobody could judge from title and abstract; reported, not counted |
| agreement and Cohen's kappa | how often the two raters agreed, and how far above chance |

In [ ]:
import sys
from pathlib import Path

import pandas as pd

REPO = Path.cwd() if (Path.cwd() / "config").is_dir() else Path.cwd().parent
sys.path.insert(0, str(REPO / "src"))
from validation import rater_agreement, resolve_final, stratum_precision, weighted_precision

TABLES = REPO / "outputs" / "tables"
LABELS_FILE = TABLES / "validation_labels.csv" 

sample = pd.read_csv(TABLES / "validation_sample.csv")
population = pd.read_csv(TABLES / "validation_population.csv", index_col="stratum")
population.index = population.index.str.split(" - ").str[0]   # older runs wrote "A - core G06N10"
labels = pd.read_csv(LABELS_FILE, dtype=str, keep_default_na=False)
labels["docdb_family_id"] = labels["docdb_family_id"].astype("int64")   # read as text so labels stay "1"/"0"/"?"

labels, unresolved = resolve_final(labels)
labelled = labels.merge(sample[["docdb_family_id", "stratum"]], on="docdb_family_id", how="left")

missing = labelled["stratum"].isna().sum()
assert missing == 0, f"{missing} labelled rows match no sampled family: check docdb_family_id"
if len(unresolved):
    print(f"{len(unresolved)} rows still need a final label (raters disagree or one is blank):")
    display(unresolved[["row", "docdb_family_id", "rater_1", "rater_2", "final"]])
else:
    print(f"All {len(labelled)} families have a final label.")

In [ ]:
assert unresolved.empty, f"settle the {len(unresolved)} rows listed above, then re-run both cells"

judged = labelled[labelled["final"] != ""]
per_stratum = stratum_precision(judged)
estimate, low, high = weighted_precision(per_stratum, population["families"])
agreement = rater_agreement(labels)

results = per_stratum.join(population.reindex(columns=["description", "families"]))
results.loc["weighted total", ["precision", "ci_low", "ci_high"]] = [estimate, low, high]
results.loc["weighted total", "families"] = population.loc[list(per_stratum.index), "families"].sum()
results.to_csv(TABLES / "validation_results.csv")

print(f"Weighted precision: {estimate:.1%}  (95% CI {low:.1%} to {high:.1%})")
print(f"Rater agreement: {agreement['agreement']:.1%} on {agreement['n']} families, "
      f"{agreement['disagreements']} disagreements, Cohen's kappa {agreement['kappa']:.2f}")
print(f"written: {(TABLES / 'validation_results.csv').relative_to(REPO)}")
results.style.format({"precision": "{:.1%}", "ci_low": "{:.1%}", "ci_high": "{:.1%}",
                      "families": "{:,.0f}"}, na_rep="")